In [2]:
import os
import random
import h5py

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torchvision.models as models
from torchinfo import summary
from sklearn.metrics import r2_score
import plotly.express as px


from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


import torchvision.transforms.functional as TF

import seaborn as sns
import matplotlib.pyplot as plt


sns.set_theme(style="ticks", palette="pastel", rc={"lines.linewidth": 2.5})

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

generator = torch.Generator().manual_seed(SEED)

# Set the device to use for training
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Data

In [3]:
folder_path = "models/"

# Create the folder path and checkpoints directory if they don't exist
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

if not os.path.exists(os.path.join(folder_path, "checkpoints")):
    os.makedirs(os.path.join(folder_path, "checkpoints"))

- Reading files

In [4]:
# modify so indices is the index of the dataframe
def read_hdf5_to_dataframe_with_index(h5_path="unified_parallel.h5"):
    with h5py.File(h5_path, "r") as f:
        viirs_start = f["viirs_start"][:]
        viirs_end = f["viirs_end"][:]
        rgb = f["rgb"][:]
        figures = f["figures"][:]
        indices = f["indices"][:]
        iso3 = f["iso3"][:]
        types = f["type"][:]

    # Decode bytes to strings for iso3
    iso3_decoded = [x.decode("utf-8") if isinstance(x, bytes) else x for x in iso3]
    types_decoded = [x.decode("utf-8") if isinstance(x, bytes) else x for x in types]

    # Create a DataFrame with indices as the index
    df = pd.DataFrame(
        {
            "viirs_start": list(viirs_start),
            "viirs_end": list(viirs_end),
            "rgb": list(rgb),
            "figures": figures,
            "iso3": iso3_decoded,
            "type": types_decoded,
        },
        index=indices,
    )

    df.sort_index(inplace=True)  # Ensure indices are sorted

    return df

In [5]:
path = "../src/data/processed/disaster.h5"
df_disaster = read_hdf5_to_dataframe_with_index(path)

path_idu = "../src/data/processed/testing.h5"
df_idu = read_hdf5_to_dataframe_with_index(path_idu)

# load csv iso3 embeddings
iso3_embeddings = pd.read_csv("../src/data/processed/embeddings_mapped.csv", index_col=0)

# combine the two dataframes
df = pd.concat([df_disaster, df_idu], ignore_index=True)

del df_disaster
del df_idu

In [6]:
len(df)

16896

In [7]:
iso3_embeddings.head()

,emb_0,emb_1,emb_2,emb_3
iso3_mapped,,,,
AND,-2.155526,-0.099045,1.090775,0.091681
ARE,-2.354902,-0.086402,1.196469,0.499288
AFG,2.381075,-0.013857,-0.791427,0.159821
ATG,-0.600651,-0.696483,-1.296734,1.598701
ALB,0.138654,-0.234058,-0.686938,-0.162994


In [8]:
import numpy as np
import plotly.express as px
import pycountry # You may need to: pip install pycountry

def plot_iso3_distribution_nature(df):
    # 1. Aggregate data
    iso3_counts = df["iso3"].value_counts().reset_index()
    iso3_counts.columns = ["iso3", "count"]
    iso3_counts["log_count"] = np.log1p(iso3_counts["count"])
    
    # 2. Map ISO3 to Country Names
    def get_country_name(code):
        try:
            return pycountry.countries.get(alpha_3=code).name
        except:
            return code # Fallback to code if not found

    iso3_counts["country_name"] = iso3_counts["iso3"].apply(get_country_name)
    
    # Get top 10 for the bar chart
    top_10 = iso3_counts.nlargest(10, "count").sort_values("count")

    # --- Figure 1: Choropleth (Keep ISO3 for locations) ---
    fig_map = px.choropleth(
        iso3_counts,
        locations="iso3",
        color="log_count",
        hover_name="country_name", # Changed to country name for better UX
        hover_data={"count": True, "log_count": False, "iso3": False},
        color_continuous_scale="Greys",
    )

    fig_map.update_geos(
        projection_type="natural earth",
        showframe=False,
        showcoastlines=False,
        showland=True,
        landcolor="rgb(240,240,240)",
        bgcolor="white"
    )

    fig_map.update_layout(
        template="simple_white",
        height=560,
        width=1000,
        margin=dict(l=10, r=10, t=60, b=10),
        font=dict(family="Arial", size=11, color="black"),
        title=dict(
            text="Climate-related disaster concentration by country<br><sup>Global distribution</sup>",
            x=0.01,
            xanchor="left",
            font=dict(size=14)
        ),
        coloraxis_colorbar=dict(
            title="Log events",
            thickness=10,
            len=0.6,
            outlinewidth=0
        ),
    )

    fig_map.write_image("distribution_map.pdf", engine="kaleido")
    fig_map.show()

    # --- Figure 2: Bar chart (Use country_name for Y-axis) ---
    fig_bar = px.bar(
        top_10,
        y="country_name", # Changed from "iso3"
        x="count",
        orientation="h",
        color_discrete_sequence=["#4D4D4D"],
    )

    fig_bar.update_traces(
        marker_line_width=0,
        opacity=0.9
    )

    fig_bar.update_layout(
        template="simple_white",
        height=480,
        width=700,
        margin=dict(l=50, r=10, t=60, b=20), # Increased left margin for long names
        font=dict(family="Arial", size=11, color="black"),
        title=dict(
            text="Top countries by climate-related disasters",
            x=0.01,
            xanchor="left",
            font=dict(size=14)
        ),
        xaxis=dict(
            title="Events",
            showgrid=True,
            gridcolor="rgba(0,0,0,0.05)",
            zeroline=False
        ),
        yaxis=dict(
            title="",
            showgrid=False,
            autorange="reversed" # Optional: keeps highest at the top
        ),
        showlegend=False
    )

    fig_bar.write_image("top_10_bar.pdf", engine="kaleido")
    fig_bar.show()

    # --- Summary ---
    share = top_10["count"].sum() / iso3_counts["count"].sum() * 100
    print(f"Top 10 countries account for {share:.1f}% of events")
    print(", ".join(top_10["country_name"].tolist()))

plot_iso3_distribution_nature(df)

/var/folders/ys/5gn1z90x7c1_khtjpp5kg6580000gn/T/ipykernel_28599/3615676146.py:62: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig_map.write_image("distribution_map.pdf", engine="kaleido")


/var/folders/ys/5gn1z90x7c1_khtjpp5kg6580000gn/T/ipykernel_28599/3615676146.py:105: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig_bar.write_image("top_10_bar.pdf", engine="kaleido")


Top 10 countries account for 54.5% of events
Sri Lanka, Malaysia, Philippines, United States, Burundi, India, Indonesia, Brazil, Somalia, Peru


# Simple baseline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor

# =========================================================
# CONFIG
# =========================================================
RANDOM_STATE = 42
TEST_SIZE = 0.1
BOOTSTRAP_ITERS = 1000

# =========================================================
# 1. FEATURE ENGINEERING MODULE
# =========================================================
def get_image_stats(image_array):
    """Extracts 8 statistical moments from an image array."""
    if image_array is None or len(image_array) == 0:
        return np.zeros(8)
    try:
        img = np.array(image_array)
        if img.size == 0: return np.zeros(8)
        flat = img.flatten()
        return np.array([
            np.mean(flat), np.std(flat), np.min(flat), np.max(flat),
            np.percentile(flat, 25), np.percentile(flat, 50),
            np.percentile(flat, 75), np.ptp(flat)
        ])
    except:
        return np.zeros(8)

def process_image_columns(df, columns_map):
    """Processes multiple image columns into a single feature DataFrame."""
    all_stats = []
    for col, prefix in columns_map.items():
        stats = df[col].apply(get_image_stats)
        stat_df = pd.DataFrame(np.array(stats.tolist()), 
                               columns=[f"{prefix}_{i}" for i in range(8)])
        all_stats.append(stat_df)
    return pd.concat(all_stats, axis=1)

def prepare_features(df, iso3_embeddings):
    """Merges embeddings, creates dummies, and builds image features."""
    embedding_df = iso3_embeddings.reset_index().rename(
        columns={iso3_embeddings.index.name or "index": "iso3"}
    )
    merged = df.merge(embedding_df, on="iso3", how="inner")
    
    type_dummies = pd.get_dummies(merged["type"], prefix="type")
    
    img_map = {"viirs_start": "vs", "viirs_end": "ve", "rgb": "rgb"}
    img_features = process_image_columns(merged, img_map)
    
    embed_cols = [c for c in embedding_df.columns if c != "iso3"]
    
    X_full_df = pd.concat([merged[embed_cols], type_dummies, img_features], axis=1)
    y = merged["figures"].to_numpy(np.float32)
    
    return X_full_df, y, embed_cols

# =========================================================
# 2. MODELING MODULE
# =========================================================
def get_xgb_regressor():
    """Returns a pre-configured XGBoost model."""
    return XGBRegressor(
        objective="reg:squarederror",
        n_estimators=5000,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        early_stopping_rounds=100
    )

def run_bootstrap_analysis(y_true, y_pred, n_iterations=BOOTSTRAP_ITERS):
    """Calculates distribution of metrics via bootstrapping."""
    stats = {"R2": [], "MAE": [], "RMSE": []}
    rng = np.random.default_rng(RANDOM_STATE)
    for _ in range(n_iterations):
        idx = rng.choice(len(y_true), size=len(y_true), replace=True)
        y_t, y_p = y_true[idx], y_pred[idx]
        stats["R2"].append(r2_score(y_t, y_p))
        stats["MAE"].append(mean_absolute_error(y_t, y_p))
        stats["RMSE"].append(np.sqrt(mean_squared_error(y_t, y_p)))
    return pd.DataFrame(stats)

# =========================================================
# 3. VISUALIZATION MODULE
# =========================================================
def format_feature_labels(feature_names):
    """Converts internal codes to human-readable labels."""
    stat_map = {0:"mean", 1:"std", 2:"min", 3:"max", 4:"25th", 5:"50th", 6:"75th", 7:"ptp"}
    labels = []
    for f in feature_names:
        if f.startswith("type_"): labels.append(f"Type: {f[5:]}")
        elif any(f.startswith(p) for p in ["vs_","ve_","rgb_"]):
            p, i = f.split("_")
            p_map = {"vs":"VIIRS Start", "ve":"VIIRS End", "rgb":"RGB"}
            labels.append(f"{p_map[p]} {stat_map[int(i)]}")
        else: labels.append(f"Embed: {f}")
    return labels

def export_performance_pdf(y_test, pred_dict, boot_dict, imp_df, output_path="Analysis_Report.pdf"):
    """Generates the 4-panel diagnostic plot and saves to PDF."""
    plt.rcParams.update({"font.family": "Arial", "font.size": 9})
    fig = plt.figure(figsize=(16, 5), dpi=300)
    gs = GridSpec(1, 4, width_ratios=[1, 1, 0.8, 0.9])

    colors = ["#4C72B0", "#55A868"]
    for i, (name, y_pred) in enumerate(pred_dict.items()):
        ax = fig.add_subplot(gs[0, i])
        ax.scatter(y_test, y_pred, s=20, alpha=0.4, color=colors[i], edgecolors='none')
        ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "--k", lw=1)
        ax.set_title(name, weight='bold')
        ax.set_xlabel("Observed (log)")
        ax.set_ylabel("Predicted (log)")
        ax.grid(True, linestyle="--", alpha=0.2)

    ax_boot = fig.add_subplot(gs[0, 2])
    v = ax_boot.violinplot([boot_dict["A"]["R2"], boot_dict["B"]["R2"]], showmedians=True)
    for i, pc in enumerate(v['bodies']):
        pc.set_facecolor(colors[i]); pc.set_edgecolor('black'); pc.set_alpha(0.6)
    ax_boot.set_xticks([1, 2]); ax_boot.set_xticklabels(["Model A", "Model B"])
    ax_boot.set_title("Statistical Stability ($R^2$)", weight='bold')

    ax_imp = fig.add_subplot(gs[0, 3])
    top_imp = imp_df.head(12).iloc[::-1]
    ax_imp.barh(top_imp["display_name"], top_imp["importance_mean"], color="#C44E52")
    ax_imp.set_title("Feature Importance", weight='bold')
    ax_imp.set_xlabel("Drop in $R^2$")

    plt.tight_layout()
    plt.savefig(output_path, format='pdf', bbox_inches='tight')
    # plt.close()
    print(f"Report successfully saved to: {output_path}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor

# =========================================================
# CONFIG
# =========================================================
RANDOM_STATE = 42
TEST_SIZE = 0.1
BOOTSTRAP_ITERS = 1000

# =========================================================
# 1. FEATURE ENGINEERING MODULE
# =========================================================
def get_image_stats(image_array):
    """Extracts 8 statistical moments from an image array."""
    if image_array is None or len(image_array) == 0:
        return np.zeros(8)
    try:
        img = np.array(image_array)
        if img.size == 0: return np.zeros(8)
        flat = img.flatten()
        return np.array([
            np.mean(flat), np.std(flat), np.min(flat), np.max(flat),
            np.percentile(flat, 25), np.percentile(flat, 50),
            np.percentile(flat, 75), np.ptp(flat)
        ])
    except:
        return np.zeros(8)

def process_image_columns(df, columns_map):
    """Processes multiple image columns into a single feature DataFrame."""
    all_stats = []
    for col, prefix in columns_map.items():
        stats = df[col].apply(get_image_stats)
        stat_df = pd.DataFrame(np.array(stats.tolist()), 
                               columns=[f"{prefix}_{i}" for i in range(8)])
        all_stats.append(stat_df)
    return pd.concat(all_stats, axis=1)

def prepare_features(df, iso3_embeddings):
    """Merges embeddings, creates dummies, and builds image features."""
    embedding_df = iso3_embeddings.reset_index().rename(
        columns={iso3_embeddings.index.name or "index": "iso3"}
    )
    merged = df.merge(embedding_df, on="iso3", how="inner")
    
    type_dummies = pd.get_dummies(merged["type"], prefix="type")
    
    img_map = {"viirs_start": "vs", "viirs_end": "ve", "rgb": "rgb"}
    img_features = process_image_columns(merged, img_map)
    
    embed_cols = [c for c in embedding_df.columns if c != "iso3"]
    
    X_full_df = pd.concat([merged[embed_cols], type_dummies, img_features], axis=1)
    y = merged["figures"].to_numpy(np.float32)
    
    return X_full_df, y, embed_cols

# =========================================================
# 2. MODELING MODULE
# =========================================================
def get_xgb_regressor():
    """Returns a pre-configured XGBoost model."""
    return XGBRegressor(
        objective="reg:squarederror",
        n_estimators=5000,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        early_stopping_rounds=100
    )

def run_bootstrap_analysis(y_true, y_pred, n_iterations=BOOTSTRAP_ITERS):
    """Calculates distribution of metrics via bootstrapping."""
    stats = {"R2": [], "MAE": [], "RMSE": []}
    rng = np.random.default_rng(RANDOM_STATE)
    for _ in range(n_iterations):
        idx = rng.choice(len(y_true), size=len(y_true), replace=True)
        y_t, y_p = y_true[idx], y_pred[idx]
        stats["R2"].append(r2_score(y_t, y_p))
        stats["MAE"].append(mean_absolute_error(y_t, y_p))
        stats["RMSE"].append(np.sqrt(mean_squared_error(y_t, y_p)))
    return pd.DataFrame(stats)

# =========================================================
# 3. VISUALIZATION MODULE
# =========================================================
def format_feature_labels(feature_names):
    """Converts internal codes to human-readable labels."""
    stat_map = {0:"mean", 1:"std", 2:"min", 3:"max", 4:"25th", 5:"50th", 6:"75th", 7:"ptp"}
    labels = []
    for f in feature_names:
        if f.startswith("type_"): labels.append(f"Type: {f[5:]}")
        elif any(f.startswith(p) for p in ["vs_","ve_","rgb_"]):
            p, i = f.split("_")
            p_map = {"vs":"VIIRS Start", "ve":"VIIRS End", "rgb":"RGB"}
            labels.append(f"{p_map[p]} {stat_map[int(i)]}")
        else: labels.append(f"Embed: {f}")
    return labels

def export_performance_pdf(y_test, pred_dict, boot_dict, imp_df, output_path="Analysis_Report.pdf"):
    """Generates the 4-panel diagnostic plot with error bars and saves to PDF."""
    plt.rcParams.update({"font.family": "Arial", "font.size": 9})
    fig = plt.figure(figsize=(16, 5), dpi=300)
    gs = GridSpec(1, 4, width_ratios=[1, 1, 0.8, 0.9])

    colors = ["#4C72B0", "#55A868"]
    
    # Panels 1 & 2: Scatters
    for i, (name, y_pred) in enumerate(pred_dict.items()):
        ax = fig.add_subplot(gs[0, i])
        ax.scatter(y_test, y_pred, s=20, alpha=0.4, color=colors[i], edgecolors='none')
        ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "--k", lw=1)
        ax.set_title(name, weight='bold')
        ax.set_xlabel("Observed (log)")
        ax.set_ylabel("Predicted (log)")
        ax.grid(True, linestyle="--", alpha=0.2)

    # Panel 3: R2 Stability
    ax_boot = fig.add_subplot(gs[0, 2])
    v = ax_boot.violinplot([boot_dict["A"]["R2"], boot_dict["B"]["R2"]], showmedians=True)
    for i, pc in enumerate(v['bodies']):
        pc.set_facecolor(colors[i]); pc.set_edgecolor('black'); pc.set_alpha(0.6)
    ax_boot.set_xticks([1, 2]); ax_boot.set_xticklabels(["Model A", "Model B"])
    ax_boot.set_title("Statistical Stability ($R^2$)", weight='bold')

    # Panel 4: Feature Importance with Error Bars
    ax_imp = fig.add_subplot(gs[0, 3])
    top_imp = imp_df.head(12).iloc[::-1]
    
    ax_imp.barh(top_imp["display_name"], 
                top_imp["importance_mean"], 
                xerr=top_imp["importance_std"], 
                color="#C44E52",
                alpha=0.8,
                error_kw={'capsize': 2, 'elinewidth': 0.8})
    
    ax_imp.set_title("Feature Importance", weight='bold')
    ax_imp.set_xlabel("Drop in $R^2$")
    ax_imp.grid(True, axis='x', linestyle="--", alpha=0.2)

    plt.tight_layout()
    plt.savefig(output_path, format='pdf', bbox_inches='tight')
    plt.close()
    print(f"Report successfully saved to: {output_path}")

# =========================================================
# MAIN EXECUTION
# =========================================================
if __name__ == "__main__":
    # Ensure variables 'df' and 'iso3_embeddings' are defined in your environment
    
    # 1. Feature Engineering
    X_full, y, embed_cols = prepare_features(df, iso3_embeddings)
    y_log = np.log1p(y)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_full, y_log, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    # 2. Training
    model_a = get_xgb_regressor()
    model_a.fit(X_train[embed_cols], y_train, 
                eval_set=[(X_test[embed_cols], y_test)], verbose=False)
    
    model_b = get_xgb_regressor()
    model_b.fit(X_train, y_train, 
                eval_set=[(X_test, y_test)], verbose=False)

    # 3. Evaluation & Bootstrapping
    preds_a = model_a.predict(X_test[embed_cols])
    preds_b = model_b.predict(X_test)
    
    boot_a = run_bootstrap_analysis(y_test, preds_a)
    boot_b = run_bootstrap_analysis(y_test, preds_b)

    # 4. Importance Calculation with Std Dev
    imp = permutation_importance(model_b, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE)
    
    imp_df = pd.DataFrame({
        "feature": X_full.columns,
        "importance_mean": imp.importances_mean,
        "importance_std": imp.importances_std
    }).sort_values("importance_mean", ascending=False)
    
    imp_df["display_name"] = format_feature_labels(imp_df["feature"])

    # 5. PDF Export
    export_performance_pdf(
        y_test = y_test,
        pred_dict = {"Model A: Embeddings": preds_a, "Model B: Combined": preds_b},
        boot_dict = {"A": boot_a, "B": boot_b},
        imp_df = imp_df,
        output_path = "Model_Performance_Analysis.pdf"
    )

Report successfully saved to: Analysis_Report.pdf


# NN

In [15]:
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# =========================================================
# CONFIG & DEVICE
# =========================================================
RANDOM_STATE = 42
TEST_SIZE    = 0.10
VAL_SIZE     = 0.10
BATCH_SIZE   = 32
EPOCHS       = 50
LR           = 3e-4
VIIRS_SIZE   = 64
RGB_SIZE     = 224
IMG_DIM      = 64          

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Using compute device: {DEVICE}")
torch.manual_seed(RANDOM_STATE)


# =========================================================
# IMAGE PREPROCESSING (Fixed for NaN safety)
# =========================================================
_rgb_normalize = transforms.Normalize([0.485, 0.456, 0.406],
                                       [0.229, 0.224, 0.225])

def prep_viirs(arr, size=VIIRS_SIZE):
    # FIX 1: Safely catch None, empty objects, and NaN floats
    if arr is None or (isinstance(arr, float) and math.isnan(arr)) or (hasattr(arr, "__len__") and len(arr) == 0):
        return torch.zeros(1, size, size)
    
    arr_np = np.asarray(arr, dtype=np.float32)
    t = torch.from_numpy(arr_np)
    if t.ndim == 2:
        t = t.unsqueeze(0)
    return nn.functional.interpolate(
        t.unsqueeze(0), (size, size), mode="bilinear", align_corners=False
    ).squeeze(0)

def prep_rgb(arr, size=RGB_SIZE):
    # FIX 1: Safely catch None, empty objects, and NaN floats
    if arr is None or (isinstance(arr, float) and math.isnan(arr)) or (hasattr(arr, "__len__") and len(arr) == 0):
        return torch.zeros(3, size, size)
    
    arr_np = np.asarray(arr, dtype=np.float32)
    t = torch.from_numpy(arr_np) / 255.0
    if t.ndim == 2:
        t = t.unsqueeze(0).repeat(3, 1, 1)
    elif t.shape[-1] == 3:
        t = t.permute(2, 0, 1)
    t = nn.functional.interpolate(
        t.unsqueeze(0), (size, size), mode="bilinear", align_corners=False
    ).squeeze(0)
    return _rgb_normalize(t)


# =========================================================
# DATASET (Fixed for Lazy Loading)
# =========================================================
class DisplacementDataset(Dataset):
    def __init__(self, df, embed_cols, type_cols):
        # Data is pre-merged and pre-encoded by prepare_data()
        self.embeddings = torch.from_numpy(df[embed_cols].to_numpy(np.float32))
        self.type_vecs  = torch.from_numpy(df[type_cols].to_numpy(np.float32))
        self.targets    = torch.from_numpy(np.log1p(df["figures"].to_numpy(np.float32)))
        
        self.viirs_start_list = df["viirs_start"].tolist()
        self.viirs_end_list   = df["viirs_end"].tolist()
        self.rgb_paths        = df["rgb"].tolist()  # FIX 2: Store paths, not raw arrays
        self.length           = len(df)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # FIX 2: Lazy load images from disk to prevent OOM crashes
        rgb_path = self.rgb_paths[idx]
        rgb_arr = None
        if isinstance(rgb_path, str):
            try:
                rgb_arr = np.array(Image.open(rgb_path).convert("RGB"))
            except Exception:
                pass # Falls back to None, prep_rgb will return zeroes

        return {
            "viirs_start": prep_viirs(self.viirs_start_list[idx]),
            "viirs_end":   prep_viirs(self.viirs_end_list[idx]),
            "rgb":         prep_rgb(rgb_arr),
            "embedding":   self.embeddings[idx],
            "type_vec":    self.type_vecs[idx],
            "target":      self.targets[idx],
        }


# =========================================================
# DATA PREPARATION (Fixed for Data Leakage)
# =========================================================
def prepare_data(df, iso3_embeddings):
    """Prepares and returns datasets, dimensions, and dataloaders for both pipelines."""
    print("Preparing datasets and dataloaders...")
    
    # 1. Merge Embeddings
    embed_df = iso3_embeddings.reset_index().rename(
        columns={iso3_embeddings.index.name or "index": "iso3"}
    )
    merged = df.merge(embed_df, on="iso3", how="inner").reset_index(drop=True)
    embed_cols = [c for c in embed_df.columns if c != "iso3"]

    # 2. Split BEFORE One-Hot Encoding to prevent data leakage (FIX 4)
    train_df, test_df = train_test_split(merged, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_df, val_df  = train_test_split(train_df, test_size=VAL_SIZE, random_state=RANDOM_STATE)

    # 3. Fit OneHotEncoder strictly on Train Data
    ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    train_type_encoded = ohe.fit_transform(train_df[["type"]])
    type_cols = [f"type_{c}" for c in ohe.get_feature_names_out(["type"])]
    
    # Apply to all splits
    train_df[type_cols] = train_type_encoded
    val_df[type_cols]   = ohe.transform(val_df[["type"]])
    test_df[type_cols]  = ohe.transform(test_df[["type"]])

    # 4. Construct Datasets
    train_ds = DisplacementDataset(train_df, embed_cols, type_cols)
    val_ds   = DisplacementDataset(val_df, embed_cols, type_cols)
    test_ds  = DisplacementDataset(test_df, embed_cols, type_cols)

    def make_loader(ds, shuffle):
        return DataLoader(
            ds, batch_size=BATCH_SIZE, shuffle=shuffle, 
            num_workers=0, pin_memory=False, persistent_workers=False
        )

    loaders = {
        "train": make_loader(train_ds, shuffle=True),
        "val":   make_loader(val_ds, shuffle=False),
        "test":  make_loader(test_ds, shuffle=False)
    }
    
    dims = {"embed_dim": len(embed_cols), "type_dim": len(type_cols)}
    return loaders, dims


# =========================================================
# ENCODERS & MODELS (Fixed for BatchNorm)
# =========================================================
class VIIRSEncoder(nn.Module):
    def __init__(self, out_dim=IMG_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, out_dim),
            nn.ReLU(),
        )
    def forward(self, x): return self.net(x)

class RGBEncoder(nn.Module):
    def __init__(self, out_dim=IMG_DIM):
        super().__init__()
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        self.features = backbone.features
        self.pool     = backbone.avgpool
        
        for p in self.features.parameters():
            p.requires_grad = False
            
        self.proj = nn.Sequential(nn.Linear(1280, out_dim), nn.ReLU())

    def forward(self, x):
        x = self.pool(self.features(x)).flatten(1)
        return self.proj(x)
        
    def train(self, mode=True):
        """FIX 5: Override train() to lock BatchNorm layers in eval mode"""
        super().train(mode)
        self.features.eval()

class EmbeddingOnlyModel(nn.Module):
    def __init__(self, embed_dim, type_dim):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(embed_dim + type_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),                   nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, batch):
        x = torch.cat([batch["embedding"], batch["type_vec"]], dim=1)
        return self.head(x).squeeze(1)

class ImageryModel(nn.Module):
    def __init__(self, embed_dim, type_dim):
        super().__init__()
        self.viirs_enc = VIIRSEncoder()
        self.rgb_enc   = RGBEncoder()
        in_dim = embed_dim + type_dim + IMG_DIM * 3
        self.head = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128),    nn.ReLU(),
            nn.Linear(128, 1),
        )
    def forward(self, batch):
        vs  = self.viirs_enc(batch["viirs_start"])
        ve  = self.viirs_enc(batch["viirs_end"])
        rgb = self.rgb_enc(batch["rgb"])
        x   = torch.cat([batch["embedding"], batch["type_vec"], vs, ve, rgb], dim=1)
        return self.head(x).squeeze(1)


# =========================================================
# ENHANCED TRAINING LOOP (Fixed for NoneType crashes)
# =========================================================
def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR):
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.MSELoss()

    model.to(DEVICE)
    
    # FIX 3: Initialize best_state cleanly to prevent crash if training fails instantly
    best_val_loss = float("inf")
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f"{'Epoch':<8} | {'Train Loss':<12} | {'Val Loss':<12} | {'LR':<10} | {'Time (s)'}")
    print("-" * 65)

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()
        model.train()
        train_losses = []
        
        for batch in train_loader:
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            loss  = criterion(model(batch), batch["target"])
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_losses.append(loss.item())

        train_loss = np.mean(train_losses)
        val_loss = _val_loss(model, val_loader, criterion)
        current_lr = scheduler.get_last_lr()[0]
        scheduler.step()
        
        elapsed = time.time() - epoch_start

        # Track best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            save_indicator = "*"
        else:
            save_indicator = " "

        if epoch % 5 == 0 or epoch == 1:
            print(f"{epoch:>3}/{epochs}{save_indicator} | {train_loss:<12.4f} | {val_loss:<12.4f} | {current_lr:<10.2e} | {elapsed:.2f}s")

    print("-" * 65)
    model.load_state_dict(best_state)
    return model

def _val_loss(model, loader, criterion):
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            losses.append(criterion(model(batch), batch["target"]).item())
    return float(np.mean(losses))


# =========================================================
# EVALUATION & PLOTTING
# =========================================================
def get_predictions(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            preds.append(model(batch).cpu().numpy())
            targets.append(batch["target"].cpu().numpy())
    return np.concatenate(targets), np.concatenate(preds)

def evaluate(y_true, y_pred):
    return {
        "R2":   r2_score(y_true, y_pred),
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
    }

def plot_comparative_results(y_test, y_embed, y_full):
    plt.rcParams.update({"font.family": "Arial", "font.size": 10, "axes.linewidth": 0.8})
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9, 4.2), dpi=300)

    def style(ax):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.3)
        ax.set_axisbelow(True)

    for ax, y_pred, title, color in [
        (ax0, y_embed, "Model A: Embedding Only", "#4C72B0"),
        (ax1, y_full,  "Model B: Embedding + Imagery", "#55A868"),
    ]:
        lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
        ax.scatter(y_test, y_pred, s=18, alpha=0.65, color=color)
        ax.plot(lims, lims, "--", color="black", linewidth=1, alpha=0.7)
        ax.set_title(title)
        ax.set_xlabel("Observed (log scale)")
        ax.set_ylabel("Predicted (log scale)")
        style(ax)

    plt.tight_layout()
    plt.show()


# =========================================================
# SEPARATED PIPELINES
# =========================================================
def run_embedding_pipeline(loaders, dims):
    """Pipeline specifically for Model A (Country Embeddings Only)"""
    print("\n" + "="*50)
    print("🚀 PIPELINE A: TRAINING EMBEDDING-ONLY MODEL")
    print("="*50)
    
    model = EmbeddingOnlyModel(dims["embed_dim"], dims["type_dim"])
    trained_model = train_model(model, loaders["train"], loaders["val"])
    
    print("\nEvaluating Pipeline A...")
    y_test, y_pred = get_predictions(trained_model, loaders["test"])
    metrics = evaluate(y_test, y_pred)
    
    print("\n--- RESULTS: EMBEDDING ONLY ---")
    for k, v in metrics.items(): print(f"  {k}: {v:.4f}")
        
    return trained_model, y_test, y_pred, metrics

def run_imagery_pipeline(loaders, dims):
    """Pipeline specifically for Model B (Embeddings + VIIRS CNN + RGB)"""
    print("\n" + "="*50)
    print("🚀 PIPELINE B: TRAINING FULL IMAGERY MODEL")
    print("="*50)
    
    model = ImageryModel(dims["embed_dim"], dims["type_dim"])
    trained_model = train_model(model, loaders["train"], loaders["val"])
    
    print("\nEvaluating Pipeline B...")
    y_test, y_pred = get_predictions(trained_model, loaders["test"])
    metrics = evaluate(y_test, y_pred)
    
    print("\n--- RESULTS: EMBEDDING + IMAGERY ---")
    for k, v in metrics.items(): print(f"  {k}: {v:.4f}")
        
    return trained_model, y_test, y_pred, metrics

Using compute device: mps


In [16]:
# =========================================================
# RUNNER
# =========================================================
# # UNCOMMENT TO RUN:
loaders, dims = prepare_data(df, iso3_embeddings)
# 
# # Run Pipeline A
model_a, y_test, y_embed, metrics_a = run_embedding_pipeline(loaders, dims)
# 
# # Run Pipeline B
# model_b, y_test_check, y_full, metrics_b = run_imagery_pipeline(loaders, dims)
# 
# # Compare Results Plot
plot_comparative_results(y_test, y_embed, y_full)

Preparing datasets and dataloaders...

🚀 PIPELINE A: TRAINING EMBEDDING-ONLY MODEL
Epoch    | Train Loss   | Val Loss     | LR         | Time (s)
-----------------------------------------------------------------
  1/50  | nan          | nan          | 3.00e-04   | 3.40s
  5/50  | nan          | nan          | 2.95e-04   | 3.28s
 10/50  | nan          | nan          | 2.77e-04   | 3.69s


KeyboardInterrupt: 

- Architecture

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights


class MultiBranchCNN(nn.Module):
    """
    Multi-branch CNN for regression over:
      - RGB satellite image  (ResNet-18 backbone — faster than ResNet-50)
      - VIIRS nightlight start/end  (lightweight CNN, separate branches)
      - Country-level scalar features  (small MLP)

    Branch output dims are proportional to information content:
      RGB → 64  |  VIIRS → 32 each  |  Country → 8  →  fused: 136
    """

    # Branch output dims — reduced for speed
    _RGB_DIM     = 64
    _VIIRS_DIM   = 32
    _COUNTRY_DIM = 8

    def __init__(self, country_features: int = 4, output_dim: int = 1, dropout: float = 0.2):
        super().__init__()
        self.output_dim = output_dim

        fuse_dim = self._RGB_DIM + self._VIIRS_DIM * 2 + self._COUNTRY_DIM  # 136

        # --- RGB backbone (ResNet-18, 3-4x faster than ResNet-50) ---
        backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        backbone.fc = nn.Identity()
        self.rgb_backbone = backbone

        # Compress 512 → 64 (ResNet-18 outputs 512, not 2048)
        self.rgb_proj = self._mlp_block(512, [self._RGB_DIM], dropout=dropout)

        # --- VIIRS branches (separate so start/end roles are distinct) ---
        self.viirs_start = self._viirs_branch()
        self.viirs_end   = self._viirs_branch()

        # --- Country projection (4 scalars → 16, no upscale bloat) ---
        self.country_proj = self._mlp_block(country_features, [self._COUNTRY_DIM], dropout=0.0)

        # --- Fusion MLP ---
        self.mlp = nn.Sequential(
            *self._mlp_block(fuse_dim, [128, 64], dropout=dropout).children(),
            nn.Linear(64, output_dim),
        )

        self._init_weights()

    # ------------------------------------------------------------------
    # Builders
    # ------------------------------------------------------------------

    @staticmethod
    def _mlp_block(in_dim: int, hidden_dims: list[int], dropout: float = 0.3) -> nn.Sequential:
        """Linear → BN → ReLU (→ Dropout) stack. No dropout on last layer."""
        layers = []
        for i, out_dim in enumerate(hidden_dims):
            layers += [nn.Linear(in_dim, out_dim), nn.BatchNorm1d(out_dim), nn.ReLU()]
            if dropout > 0 and i < len(hidden_dims) - 1:
                layers.append(nn.Dropout(dropout))
            in_dim = out_dim
        return nn.Sequential(*layers)

    @classmethod
    def _viirs_branch(cls) -> nn.Sequential:
        """Small CNN for single-channel nightlight tiles → (VIIRS_DIM,)."""
        def conv_bn_relu(in_c, out_c):
            return [nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU()]

        return nn.Sequential(
            *conv_bn_relu(1, 32),
            *conv_bn_relu(32, 32),
            nn.MaxPool2d(2),
            *conv_bn_relu(32, cls._VIIRS_DIM),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )

    def _init_weights(self):
        """Kaiming Normal for all custom Linear / Conv2d layers."""
        custom = [self.rgb_proj, self.viirs_start, self.viirs_end,
                  self.country_proj, self.mlp]
        for block in custom:
            for m in block.modules():
                if isinstance(m, (nn.Linear, nn.Conv2d)):
                    nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

    # ------------------------------------------------------------------
    # Forward
    # ------------------------------------------------------------------

    def forward(
        self,
        rgb_img: torch.Tensor,           # (B, 3, H, W)
        viirs_start: torch.Tensor,       # (B, 1, H', W')
        viirs_end: torch.Tensor,         # (B, 1, H', W')
        country_embedding: torch.Tensor, # (B, country_features)
    ) -> torch.Tensor:
        rgb     = self.rgb_proj(self.rgb_backbone(rgb_img))   # (B, 128)
        v_start = self.viirs_start(viirs_start)               # (B,  64)
        v_end   = self.viirs_end(viirs_end)                   # (B,  64)
        country = self.country_proj(country_embedding)         # (B,  16)

        fused  = torch.cat([rgb, v_start, v_end, country], dim=1)  # (B, 272)
        output = self.mlp(fused)                                    # (B, output_dim)
        return output

In [ ]:
batch_size = 128  # Increased from 64 — better GPU utilization (MPS benefits from larger batches)
learning_rate = 5e-4  # Slightly lower LR for larger batch size
num_epochs = 40

# Training speed flags
USE_GRADIENT_CHECKPOINTING = False
USE_TORCH_COMPILE = False  # Set to True if PyTorch 2.0+
INCREASE_BATCH_SIZE = True  # Already done above (128)

- Transform

In [ ]:
class JointTransform:
    def __init__(self, target_size=(224, 224), train: bool = True,
                 viirs_mean: float = 0.42, viirs_std: float = 0.81):
        self.train = train
        self.viirs_mean = viirs_mean
        self.viirs_std  = viirs_std

        self.resize_rgb = transforms.Resize(
            target_size,
            interpolation=transforms.InterpolationMode.BILINEAR,
            antialias=True,
        )
        self.color_jitter = transforms.ColorJitter(
            brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1
        )
        self.normalize_rgb = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )

    def _normalize_viirs(self, t: torch.Tensor) -> torch.Tensor:
        t = torch.log1p(t.float())
        return (t - self.viirs_mean) / self.viirs_std

    def __call__(self, rgb, viirs_start, viirs_end):
        # rgb is PIL at this point; viirs_* are already tensors
        rgb = self.resize_rgb(rgb)

        if self.train:
            rgb = self.color_jitter(rgb)
            if random.random() > 0.5:
                rgb, viirs_start, viirs_end = map(TF.hflip, [rgb, viirs_start, viirs_end])
            if random.random() > 0.5:
                rgb, viirs_start, viirs_end = map(TF.vflip, [rgb, viirs_start, viirs_end])

        rgb = self.normalize_rgb(rgb)
        viirs_start = self._normalize_viirs(viirs_start)
        viirs_end   = self._normalize_viirs(viirs_end)

        return rgb, viirs_start, viirs_end

- Dataset and DataLoader

In [ ]:
class IDPDataset(Dataset):
    def __init__(self, df, joint_transform=None, target_col="figures", target_transform=None):
        self.joint_transform = joint_transform
        self.target_transform = target_transform

        # Pre-extract to numpy once — avoids repeated iloc overhead
        self.viirs_start  = df["viirs_start"].to_numpy()
        self.viirs_end    = df["viirs_end"].to_numpy()
        self.rgb          = df["rgb"].to_numpy()
        self.targets      = torch.tensor(df[target_col].values,    dtype=torch.float32)
        self.iso3_encoded = torch.tensor(
            np.stack(df["iso3_encoded"].values), dtype=torch.float32
        )  # (N, 3)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        # VIIRS: stored as (H, W) → (1, H, W)
        viirs_start = torch.as_tensor(self.viirs_start[idx]).float().unsqueeze(0)
        viirs_end   = torch.as_tensor(self.viirs_end[idx]).float().unsqueeze(0)

        # RGB: stored as (H, W, C) uint8 → (C, H, W) float in [0, 1]
        rgb = torch.as_tensor(self.rgb[idx]).float() / 255.0
        rgb = rgb.permute(2, 0, 1)  # (H, W, 3) → (3, H, W)

        if self.joint_transform:
            rgb, viirs_start, viirs_end = self.joint_transform(rgb, viirs_start, viirs_end)

        target = self.targets[idx]
        if self.target_transform:
            target = self.target_transform(target)

        return rgb, viirs_start, viirs_end, target.unsqueeze(0), self.iso3_encoded[idx]

In [ ]:
# ── 1. Build iso3 → embedding lookup ─────────────────────────────────────────
embedding_key = None
for candidate in ("iso3", "iso3_mapped"):
    if candidate in iso3_embeddings.columns:
        embedding_key = candidate
        break

if embedding_key is None and getattr(iso3_embeddings.index, "name", None) in ("iso3", "iso3_mapped"):
    emb_df = iso3_embeddings.copy()
else:
    emb_df = iso3_embeddings.copy().reset_index()
    index_name = iso3_embeddings.index.name or "index"
    if index_name not in emb_df.columns:
        emb_df = emb_df.rename(columns={index_name: "iso3"})
    if embedding_key is None and "iso3" not in emb_df.columns and "iso3_mapped" in emb_df.columns:
        emb_df = emb_df.rename(columns={"iso3_mapped": "iso3"})
    if "iso3" not in emb_df.columns and "iso3_mapped" not in emb_df.columns:
        raise KeyError("Could not find an ISO3 identifier column in iso3_embeddings")

if embedding_key is None:
    emb_df = emb_df.rename_axis("iso3").reset_index() if getattr(emb_df.index, "name", None) in ("iso3", "iso3_mapped") else emb_df
    if "iso3" not in emb_df.columns and "iso3_mapped" in emb_df.columns:
        emb_df = emb_df.rename(columns={"iso3_mapped": "iso3"})
else:
    emb_df = emb_df.rename(columns={embedding_key: "iso3"})

emb_df = emb_df.set_index("iso3").select_dtypes(include=[np.number])
N_COUNTRY_FEATURES = emb_df.shape[1]


def _get_embedding(code: str) -> np.ndarray:
    try:
        row = emb_df.loc[code]
        if isinstance(row, pd.DataFrame):   # handles duplicate iso3 rows
            row = row.iloc[0]
        return np.asarray(row, dtype=np.float32).ravel()
    except KeyError:
        return np.zeros(N_COUNTRY_FEATURES, dtype=np.float32)


df["iso3_encoded"] = df["iso3"].apply(_get_embedding)

# ── 2. Train / val / test split ───────────────────────────────────────────────
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# ── 3. Target normalization (log1p + z-score using training data only) ─────────
target_log_train = np.log1p(train_df["figures"].astype(np.float32).to_numpy())
TARGET_LOG_MEAN = float(target_log_train.mean())
TARGET_LOG_STD = float(target_log_train.std()) + 1e-6


def _normalize_target(target):
    target_tensor = torch.as_tensor(target, dtype=torch.float32)
    return (torch.log1p(target_tensor) - TARGET_LOG_MEAN) / TARGET_LOG_STD


def _denormalize_target(target):
    target_array = np.asarray(target, dtype=np.float32)
    return np.expm1(target_array * TARGET_LOG_STD + TARGET_LOG_MEAN)


# ── 4. VIIRS normalisation stats (training data only — no leakage) ────────────
viirs_log = np.concatenate([
    np.log1p(np.stack(train_df["viirs_start"].values).ravel()),
    np.log1p(np.stack(train_df["viirs_end"].values).ravel()),
])
VIIRS_MEAN = float(viirs_log.mean())
VIIRS_STD  = float(viirs_log.std()) + 1e-6

# ── 5. Transforms ─────────────────────────────────────────────────────────────
train_transform = JointTransform(
    target_size=(224, 224),
    train=True,
    viirs_mean=VIIRS_MEAN,
    viirs_std=VIIRS_STD,
)
eval_transform = JointTransform(
    target_size=(224, 224),
    train=False,
    viirs_mean=VIIRS_MEAN,
    viirs_std=VIIRS_STD,
)

# ── 6. Datasets ───────────────────────────────────────────────────────────────
train_dataset = IDPDataset(
    df=train_df,
    joint_transform=train_transform,
    target_transform=_normalize_target,
)
val_dataset = IDPDataset(
    df=val_df,
    joint_transform=eval_transform,
    target_transform=_normalize_target,
)
test_dataset = IDPDataset(
    df=test_df,
    joint_transform=eval_transform,
    target_transform=_normalize_target,
)

# ── 7. Model ──────────────────────────────────────────────────────────────────
model = MultiBranchCNN(country_features=N_COUNTRY_FEATURES, output_dim=1)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=generator, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

In [ ]:
model = MultiBranchCNN(output_dim=1).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Optional: gradient checkpointing for reduced memory (ResNet-18 backbone)
if USE_GRADIENT_CHECKPOINTING:
    for module in model.rgb_backbone.modules():
        if hasattr(module, "gradient_checkpointing"):
            module.gradient_checkpointing = True

# Optional: torch.compile for PyTorch 2.0+
if USE_TORCH_COMPILE:
    try:
        model = torch.compile(model)
    except Exception:
        pass

In [ ]:
summary(
    model,
    input_size=(
        ( batch_size, 3, 224, 224),
        ( batch_size, 1, 20, 20),
        ( batch_size, 1, 20, 20),
        ( batch_size, 4),
    ),
    device=device.type,
)

In [ ]:
# show one batch of data to verify everything is working
for rgb, viirs_start, viirs_end, targets, iso3_emb in train_loader:
    print("RGB batch shape:", rgb.shape)  # (B, 3, H, W)
    print("VIIRS start batch shape:", viirs_start.shape)  # (B, 1, H', W')
    print("VIIRS end batch shape:", viirs_end.shape)  # (B, 1, H', W')
    print("Country embedding batch shape:", iso3_emb.shape)  # (B, country_features)
    print("Targets batch shape:", targets.shape)  # (B,)
    break

# Training

In [ ]:
train_size = len(train_loader.dataset)
val_size = len(val_loader.dataset)
test_size = len(test_loader.dataset)
print(f"Training samples: {train_size}, Validation samples: {val_size}, Test samples: {test_size}")

In [ ]:
best_loss = float("inf")
patience = 10
patience_counter = 0
train_losses = []
val_losses = []

print(f"Training on {train_size} examples, validating on {val_size} examples")
print(f"Device: {device}")

for epoch in range(num_epochs):
    # Training phase
    model.train()
    epoch_train_loss = 0.0

    for rgb, viirs_start, viirs_end, targets, iso3_emb in train_loader:
        # Move data to device with non_blocking=True for async transfer
        rgb = rgb.to(device, non_blocking=True)
        viirs_start = viirs_start.to(device, non_blocking=True)
        viirs_end = viirs_end.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        iso3_emb = iso3_emb.to(device, non_blocking=True)

        # Forward pass
        outputs = model(rgb, viirs_start, viirs_end, iso3_emb)
        loss = criterion(outputs, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_train_loss += loss.item()

    epoch_train_loss /= len(train_loader)
    train_losses.append(epoch_train_loss)

    # Validation phase
    model.eval()
    epoch_val_loss = 0.0

    with torch.no_grad():
        for rgb, viirs_start, viirs_end, targets, iso3_emb in val_loader:
            rgb = rgb.to(device, non_blocking=True)
            viirs_start = viirs_start.to(device, non_blocking=True)
            viirs_end = viirs_end.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            iso3_emb = iso3_emb.to(device, non_blocking=True)

            outputs = model(rgb, viirs_start, viirs_end, iso3_emb)
            val_loss = criterion(outputs, targets)
            epoch_val_loss += val_loss.item()

    epoch_val_loss /= len(val_loader)
    val_losses.append(epoch_val_loss)

    # Print epoch statistics (only every 5 epochs to speed up)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(
            f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}"
        )

    # Early stopping
    if epoch_val_loss < best_loss:
        best_loss = epoch_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), folder_path + "checkpoints/best_model_gidd.pth")
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f" ✅ Saved with validation loss: {best_loss:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch + 1} epochs")
            break

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# if using ReduceLROnPlateau, restore lr
if "scheduler" in locals():
    for param_group in optimizer.param_groups:
        param_group["lr"] = learning_rate  # Reset learning rate to initial value

# Load best model

- Inference using best model

In [ ]:
model.load_state_dict(torch.load(folder_path + "checkpoints/best_model_gidd.pth"))

In [ ]:
def predict_batch(model, batch, device):
    model.eval()
    rgb, viirs_start, viirs_end, targets, iso3_emb = batch  # <-- capture targets

    rgb = rgb.to(device)
    viirs_start = viirs_start.to(device)
    viirs_end = viirs_end.to(device)
    iso3_emb = iso3_emb.to(device)
    targets = targets.to(device)

    with torch.no_grad():
        outputs = model(rgb, viirs_start, viirs_end, iso3_emb)

    return outputs.cpu().numpy(), targets.cpu().numpy()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def get_predictions_and_targets_log_space(model, dataloader, device):
    """Gets predictions and targets in log space (normalized)."""
    model.eval()
    predictions, targets = [], []

    with torch.no_grad():
        for batch in dataloader:
            batch_preds, batch_targets = predict_batch(model, batch, device)
            predictions.extend(batch_preds)
            targets.extend(batch_targets)

    # Return in log space (normalized) — do NOT denormalize
    predictions = np.array(predictions)
    targets = np.array(targets)
    return predictions, targets


# Compute R² in log space for all three datasets
print("=" * 60)
print("R² SCORES IN LOG SPACE (normalized)")
print("=" * 60)

pred_train_log, targets_train_log = get_predictions_and_targets_log_space(model, train_loader, device)
r2_train_log = r2_score(targets_train_log, pred_train_log)
print(f"Train R² (log space):      {r2_train_log:.6f}")

pred_val_log, targets_val_log = get_predictions_and_targets_log_space(model, val_loader, device)
r2_val_log = r2_score(targets_val_log, pred_val_log)
print(f"Validation R² (log space): {r2_val_log:.6f}")

pred_test_log, targets_test_log = get_predictions_and_targets_log_space(model, test_loader, device)
r2_test_log = r2_score(targets_test_log, pred_test_log)
print(f"Test R² (log space):       {r2_test_log:.6f}")

print("=" * 60)
print("ADDITIONAL METRICS (log space)")
print("=" * 60)
mae_test_log = mean_absolute_error(targets_test_log, pred_test_log)
rmse_test_log = np.sqrt(mean_squared_error(targets_test_log, pred_test_log))
print(f"Test MAE (log space):      {mae_test_log:.6f}")
print(f"Test RMSE (log space):     {rmse_test_log:.6f}")
print("=" * 60)

In [ ]:
# plot r2 scatter in log space
plt.figure(figsize=(6, 6))
plt.scatter(targets_test_log, pred_test_log, alpha=0.6, color="#4C72B0", edgecolor="k", s=50)
lims = [min(targets_test_log.min(), pred_test_log.min()), max(targets_test_log.max(), pred_test_log.max())]
plt.plot(lims, lims, "--", color="black", linewidth=1, alpha=0.7)
plt.xlabel("Observed")
plt.ylabel("Predicted")
plt.title(f"Test Set Predictions (R²={r2_test_log:.3f})")
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

In [ ]:
# plot train r2 scatter in log space
plt.figure(figsize=(6, 6))
plt.scatter(targets_train_log, pred_train_log, alpha=0.6, color="#55A868", edgecolor="k", s=50)
lims = [min(targets_train_log.min(), pred_train_log.min()), max(targets_train_log.max(), pred_train_log.max())]
plt.plot(lims, lims, "--", color="black", linewidth=1, alpha=0.7)
plt.xlabel("Observed")
plt.ylabel("Predicted")
plt.title(f"Train Set Predictions (R²={r2_train_log:.3f})")
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()